# 05. Classification Expand

memo_id 기준 `classification_detail` 결과를 원본 리뷰 row 기준 `classification_full`로 확장합니다.

In [ ]:
import sys
import importlib

PROJECT_ROOT = "/Workspace/Users/jungryo.lee@lge.com/prj_TV_voc"
SRC_ROOT = f"{PROJECT_ROOT}/src"

if SRC_ROOT not in sys.path:
    sys.path.append(SRC_ROOT)

import common.config_loader as config_loader
import taxonomy.classification_expander as classification_expander
import pipeline.run_classification_expand as run_classification_expand

importlib.reload(config_loader)
importlib.reload(classification_expander)
importlib.reload(run_classification_expand)

from common.config_loader import load_config, get_output_table
from pipeline.run_classification_expand import run_classification_expand
from pyspark.sql import functions as F
from pyspark.sql.window import Window

config = load_config(f"{PROJECT_ROOT}/config/settings.yaml")
print("config loaded")

In [ ]:
# 테스트 범위 설정
# 전체 확장 전에 특정 cate_1/cate_2/sc_measurement로 먼저 확인하세요.
TARGET_CATE_1_DEPTH = "07. 스마트 사용성"
TARGET_CATE_2_DEPTH = "07-01. 채널/컨텐츠 APP"
TARGET_SC_MEASUREMENT = 1

# 필요하면 특정 버전만 고정할 수 있습니다. None이면 저장된 최신 detail을 사용합니다.
MODEL_VERSION = None
PROMPT_VERSION = None
TAXONOMY_VERSION = None

WRITE_MODE = "replace_groups"

In [ ]:
result = run_classification_expand(
    spark,
    config=config,
    cate_1_depth=TARGET_CATE_1_DEPTH,
    cate_2_depth=TARGET_CATE_2_DEPTH,
    sc_measurement=TARGET_SC_MEASUREMENT,
    model_version=MODEL_VERSION,
    prompt_version=PROMPT_VERSION,
    taxonomy_version=TAXONOMY_VERSION,
    write_mode=WRITE_MODE,
)

result

In [ ]:
full_table = get_output_table(config, "classification_full")

full_df = (
    spark.table(full_table)
    .where(F.col("cate_1_depth") == TARGET_CATE_1_DEPTH)
    .where(F.col("cate_2_depth") == TARGET_CATE_2_DEPTH)
    .where(F.col("sc_measurement") == TARGET_SC_MEASUREMENT)
)

display(
    full_df.groupBy("pred_topic", "pred_topic_type")
    .agg(
        F.count("*").alias("raw_row_cnt"),
        F.countDistinct("memo_id").alias("distinct_memo_id_cnt"),
    )
    .withColumn("raw_row_ratio_pct", F.round(F.col("raw_row_cnt") / F.sum("raw_row_cnt").over(Window.partitionBy()) * 100, 2))
    .orderBy(F.col("raw_row_cnt").desc(), F.col("pred_topic").asc())
)

In [ ]:
display(
    full_df.select(
        "memo",
        "memo_id",
        "pred_topic",
        "pred_topic_type",
        "classification_stage",
        "match_reason",
        "year",
        "country",
        "brand_name",
        "device_type",
    )
    .orderBy(F.rand(42))
    .limit(30)
)

In [ ]:
detail_table = get_output_table(config, "classification_detail")

detail_df = (
    spark.table(detail_table)
    .where(F.col("cate_1_depth") == TARGET_CATE_1_DEPTH)
    .where(F.col("cate_2_depth") == TARGET_CATE_2_DEPTH)
    .where(F.col("sc_measurement") == TARGET_SC_MEASUREMENT)
)

validation_summary = {
    "classification_detail_rows": detail_df.count(),
    "classification_detail_distinct_memo_ids": detail_df.select("memo_id").dropDuplicates().count(),
    "classification_full_rows": full_df.count(),
    "classification_full_distinct_memo_ids": full_df.select("memo_id").dropDuplicates().count(),
}

validation_summary["raw_rows_per_classified_memo_id"] = round(
    validation_summary["classification_full_rows"] / validation_summary["classification_full_distinct_memo_ids"],
    4,
) if validation_summary["classification_full_distinct_memo_ids"] else 0

validation_summary

In [ ]:
display(
    full_df.groupBy("memo", "memo_id", "pred_topic", "pred_topic_type")
    .agg(F.count("*").alias("raw_row_cnt"))
    .orderBy(F.col("raw_row_cnt").desc(), F.col("memo").asc())
    .limit(30)
)